# C2O Research Notebook: Model Run, Output Generation, And Tests

This notebook is the code-section research implementation for the C2O coursework. It is intended to be the single runnable entry point: open the notebook, run all cells, and it will build the point-in-time panel from `data/`, train the model, generate the submitted `outputs/` files, audit those files, and run the test suite.

The default configuration uses all dates available in `prices.parquet` while freezing model development at `2024-12-31`. With the submitted data this reproduces the 2010-2024 package; if the marker replaces `data/` with later held-out files, the same notebook scores those later dates without using post-2024 labels to refit the model.

## Research Plan

1. Resolve the data folder, output folder, run cutoff, and fixed development cutoff from one notebook parameter cell.
2. Build the point-in-time stock-day panel from raw parquet/csv inputs.
3. Audit the return identity and equal-weight close/open/close decomposition.
4. Train the baseline walk-forward model and backtest the 250M portfolio.
5. Train the promoted Phase 2 expanding-window model and backtest the final portfolio.
6. Reproduce the full Phase 1 and Phase 2 model-search tables, including target, weighting, training-window, ridge, elastic-net, and short-treatment alternatives.
7. Run validation/holdout, full feature-group ablation, capacity, cost, and borrow checks.
8. Regenerate every submitted output data file directly from the notebook run.
9. Read the regenerated artefacts back, audit consistency, and run the relevant pytest checks.

In [1]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'c2o_strategy').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('Could not locate 03_Code_Repository root.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'src'
TOOLS = ROOT / 'tools'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

np.random.seed(42)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print(f'Notebook root: {ROOT.name}')
print(f'Python version: {sys.version.split()[0]}')
print(f'Platform: {platform.system()} {platform.machine()}')


Notebook root: 03_Code_Repository
Python version: 3.13.5
Platform: Darwin arm64


In [2]:
from c2o_strategy.alpha import compute_information_coefficients, summarise_ic
from c2o_strategy.config import StrategyConfig
from c2o_strategy.data import load_benchmark_returns, prepare_data
from c2o_strategy.experiments import (
    MIN_TRAINING_OBSERVATIONS,
    ExperimentSpec,
    add_target_transform,
    add_walk_forward_alpha_return,
    cache_path,
    experiment_specs,
    ic_summary,
    run_experiment_backtest,
    score_panel_with_target,
    slim_panel,
    summarise_experiment,
)
from c2o_strategy.features import (
    add_cross_sectional_ranks,
    add_point_in_time_features,
    apply_eligibility_filters,
    build_eligibility_summary,
)
from c2o_strategy.final_strategy import (
    FEATURE_GROUPS,
    aum_label,
    final_backtest_with_positions,
    final_phase2_spec,
    final_spec,
    performance_row,
    report_ready_dir,
    write_borrow_external_validation,
    write_borrow_reports,
    write_earnings_timing_examples,
    write_final_strategy_config,
    write_position_capacity_summary,
    write_short_interest_lag_examples,
    write_stylised_fact_audit,
)
from c2o_strategy.metrics import stress_window_summary
from c2o_strategy.phase2 import (
    DEFAULT_ALPHA_METHOD,
    PERIODS,
    TARGET,
    TARGET_DEFINITION,
    ic_for_period,
    learn_phase2_weights,
    load_or_train_phase2_weight_schedule,
    phase2_cache_path,
    phase2_specs,
    run_phase2_backtest,
    score_panel_phase2,
    summarise_period,
    training_start_for_window,
    write_locked_strategy_robustness,
)
from c2o_strategy.reporting import (
    compute_equal_weight_decomposition,
    generate_tearsheet,
    plot_equal_weight_decomposition,
    plot_ic,
)

# Notebook parameters. The marker can change only this block when running new data.
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
RUN_START_DATE = '2010-01-01'
RUN_CUTOFF = None  # None means use the max date available in prices.parquet.
RUN_DEVELOPMENT_CUTOFF = '2024-12-31'  # freeze fitted weights after the coursework development window.
SKIP_QUANTSTATS = False
USE_MODEL_CACHE = True
AUM = 250_000_000.0
SUBMISSION_DEVELOPMENT_CUTOFF = pd.Timestamp('2024-12-31')


def infer_max_price_date(data_dir: Path) -> str:
    dates = pd.read_parquet(data_dir / 'prices.parquet', columns=['date'])['date']
    return str(pd.to_datetime(dates).max().date())


resolved_cutoff = infer_max_price_date(DATA_DIR) if RUN_CUTOFF is None else RUN_CUTOFF
resolved_development_cutoff = RUN_DEVELOPMENT_CUTOFF or resolved_cutoff
if pd.Timestamp(resolved_development_cutoff) > pd.Timestamp(resolved_cutoff):
    resolved_development_cutoff = resolved_cutoff

config = StrategyConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    start_date=RUN_START_DATE,
    cutoff=resolved_cutoff,
    development_cutoff=resolved_development_cutoff,
)
config.output_dir.mkdir(parents=True, exist_ok=True)
config.figures_dir.mkdir(parents=True, exist_ok=True)
report_ready_dir(config)

run_context = pd.Series({
    'data_dir': str(config.data_dir.relative_to(ROOT)),
    'output_dir': str(config.output_dir.relative_to(ROOT)),
    'start_date': config.start_date,
    'run_cutoff': config.cutoff,
    'development_cutoff': config.development_cutoff,
    'uses_auto_max_price_date': RUN_CUTOFF is None,
    'heldout_mode_after_2024': config.cutoff_ts > SUBMISSION_DEVELOPMENT_CUTOFF,
    'universe_size': config.universe_size,
    'participation_cap': config.participation_cap,
    'basket_fraction': config.basket_fraction,
    'AUM_used_for_main_research_display': AUM,
    'uses_trained_model_cache_when_available': USE_MODEL_CACHE,
})
run_context

data_dir                                                data
output_dir                                           outputs
start_date                                        2010-01-01
run_cutoff                                        2024-12-31
development_cutoff                                2024-12-31
uses_auto_max_price_date                                True
heldout_mode_after_2024                                False
universe_size                                           1000
participation_cap                                   0.050000
basket_fraction                                     0.030000
AUM_used_for_main_research_display        250,000,000.000000
uses_trained_model_cache_when_available                 True
dtype: object

## 1. Raw Inputs And Fixed Assumptions

The notebook starts from the raw coursework files in `data/`. The cleaned submission keeps generated output files for audit and tests, but model training below reads the raw parquet/csv input data directly.


In [3]:
expected_data_files = [
    'prices.parquet', 'sp500_tr.parquet', 'sp500_constituents.parquet',
    'sp400_constituents.parquet', 'all_data.parquet', 'cheapness_scores.parquet',
    'earnings_calendar.parquet', 'earnings_transfo.parquet', 'short_interest_transfo.parquet',
    'piotrosky.parquet', 'gics_info.parquet', 'regime.parquet',
    'rolling_scores_upgrade.csv', 'rolling_scores_downgrade.csv',
]

file_check = pd.DataFrame({
    'file': expected_data_files,
    'exists': [(config.data_dir / name).exists() for name in expected_data_files],
    'size_mb': [
        (config.data_dir / name).stat().st_size / 1024**2 if (config.data_dir / name).exists() else np.nan
        for name in expected_data_files
    ],
})
assert file_check['exists'].all(), file_check.loc[~file_check['exists'], 'file'].tolist()
file_check


,file,exists,size_mb
0,prices.parquet,True,238.194407
1,sp500_tr.parquet,True,0.263647
2,sp500_constituents.parquet,True,0.020534
3,sp400_constituents.parquet,True,0.011790
4,all_data.parquet,True,241.975775
5,cheapness_scores.parquet,True,312.805212
6,earnings_calendar.parquet,True,0.365951
7,earnings_transfo.parquet,True,97.750225
8,short_interest_transfo.parquet,True,14.070617
9,piotrosky.parquet,True,0.145398


## 2. Build The Point-In-Time Research Panel

This is the heavy research cell. It loads raw prices and auxiliary data, constructs adjusted close/open returns, freezes the annual top-1000 universe, applies point-in-time earnings/short-interest controls, engineers features, applies trading filters, and builds daily cross-sectional ranks.


In [4]:
t0 = time.perf_counter()
prepared = prepare_data(config)
raw_panel = prepared.panel

research_panel = add_point_in_time_features(raw_panel)
research_panel = apply_eligibility_filters(research_panel, config)
research_panel, rank_columns = add_cross_sectional_ranks(research_panel)
model_panel = slim_panel(research_panel, rank_columns)

panel_elapsed = time.perf_counter() - t0
panel_summary = pd.Series({
    'stock_days_raw_panel': len(raw_panel),
    'stock_days_model_panel': len(model_panel),
    'date_min': str(pd.to_datetime(model_panel['date']).min().date()),
    'date_max': str(pd.to_datetime(model_panel['date']).max().date()),
    'ranked_features': len(rank_columns),
    'trade_eligible_stock_days': int(model_panel['is_trade_eligible'].sum()),
    'elapsed_seconds': round(panel_elapsed, 2),
})
panel_summary


stock_days_raw_panel            5470423
stock_days_model_panel          5470423
date_min                     2010-01-04
date_max                     2024-12-31
ranked_features                      28
trade_eligible_stock_days       3194145
elapsed_seconds               46.920000
dtype: object

## 3. Data And Return Sanity Checks

The return identity is checked from the raw price-derived panel. The equal-weight decomposition is also recomputed here rather than read from `outputs/equal_weight_return_decomposition.csv`.


In [5]:
prepared.reconciliation_summary


,tolerance,stock_days_checked,fail_count,fail_fraction,median_abs_residual,p99_abs_residual,max_abs_residual
0,0.000000,5469968,0,0.000000,0.000000,0.000000,0.000000


In [6]:
def annual_stats(series: pd.Series) -> dict[str, float]:
    s = series.dropna()
    return {
        'annual_return': (1.0 + s).prod() ** (252.0 / len(s)) - 1.0,
        'annual_vol': s.std(ddof=1) * np.sqrt(252.0),
        'sharpe': s.mean() / s.std(ddof=1) * np.sqrt(252.0),
        'cumulative_return': (1.0 + s).prod() - 1.0,
        'days': len(s),
    }

return_decomposition = compute_equal_weight_decomposition(raw_panel)
return_summary = pd.DataFrame({
    name: annual_stats(return_decomposition[name])
    for name in ['overnight', 'intraday', 'close_to_close']
}).T
return_summary


,annual_return,annual_vol,sharpe,cumulative_return,days
overnight,0.096600,0.117525,0.843923,2.978977,"3,774.000000"
intraday,0.045526,0.147759,0.375226,0.947889,"3,774.000000"
close_to_close,0.143707,0.194294,0.788834,6.470237,"3,774.000000"


In [7]:
universe_summary = prepared.universe_summary.copy()
eligibility_summary = build_eligibility_summary(research_panel)

print('Universe count by year:')
display(universe_summary[['year', 'universe_count', 'mid_year_exits']].head(3))
display(universe_summary[['year', 'universe_count', 'mid_year_exits']].tail(3))

print('Top eligibility reasons by stock-day count:')
display(
    eligibility_summary.groupby('eligibility_reason', as_index=False)['stock_days']
    .sum()
    .sort_values('stock_days', ascending=False)
)


Universe count by year:


,year,universe_count,mid_year_exits
0,2010,986,0
1,2011,986,0
2,2012,988,0


,year,universe_count,mid_year_exits
12,2022,988,0
13,2023,988,0
14,2024,989,0


Top eligibility reasons by stock-day count:


,eligibility_reason,stock_days
4,OK,3194145
3,MCAP_FAIL,1596321
0,ADV_FAIL,513267
2,EARN_WINDOW,116925
5,PRICE_FAIL,26307
6,VOL_FAIL,22458
1,DATA_FAIL,1000


## 4. Baseline Model Run

The baseline uses the original cross-sectional ranked overnight target, a transparent walk-forward feature-weight model, top/bottom 3% baskets, equal weighting, 5% ADV20 participation cap, and the explicit commission/slippage/borrow cost schedule.


In [8]:
baseline_spec = ExperimentSpec(
    'A_basket_size',
    'top_bottom_3pct',
    notes='Baseline rank-target model run inside this notebook.',
)

t0 = time.perf_counter()
baseline_scored = score_panel_with_target(model_panel, rank_columns, config, 'rank')
baseline_scored = add_walk_forward_alpha_return(baseline_scored, config)
baseline_daily = run_experiment_backtest(baseline_scored, AUM, config, baseline_spec)
baseline_ic = ic_summary(baseline_scored)
baseline_result = summarise_experiment(baseline_daily, baseline_spec, AUM, baseline_ic)
baseline_result['elapsed_seconds'] = round(time.perf_counter() - t0, 2)

pd.Series(baseline_result)[[
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'gross_sharpe', 'commission_drag', 'slippage_drag', 'borrow_drag',
    'average_gross_exposure_used', 'average_daily_turnover', 'IC_mean', 'IC_tstat',
    'elapsed_seconds',
]]


  scoring rank target for year 2010...


  scoring rank target for year 2011...


  scoring rank target for year 2012...


  scoring rank target for year 2013...


  scoring rank target for year 2014...


...

  scoring rank target for year 2021...


  scoring rank target for year 2022...


  scoring rank target for year 2023...


  scoring rank target for year 2024...


net_annual_return              0.016869
net_vol                        0.047099
net_sharpe                     0.378713
max_drawdown                  -0.245771
gross_sharpe                   1.728596
commission_drag                0.308959
slippage_drag                  0.928469
borrow_drag                    0.112455
average_gross_exposure_used    0.578925
average_daily_turnover         1.157850
IC_mean                        0.027312
IC_tstat                       8.362962
elapsed_seconds               20.230000
dtype: object

## 5. Promoted Phase 2 Model Run

The promoted final strategy is trained here, from the same point-in-time feature panel. It changes the target to volatility-scaled next overnight return and uses expanding-window transparent feature-weight estimation.


In [9]:
phase2_final_spec = final_phase2_spec()

t0 = time.perf_counter()
final_scored = score_panel_phase2(
    model_panel,
    rank_columns,
    config,
    phase2_final_spec.training_window,
    phase2_final_spec.alpha_learning_method,
    use_model_cache=USE_MODEL_CACHE,
)
final_daily, final_positions = final_backtest_with_positions(final_scored, AUM, config)
final_ic = ic_summary(final_scored)
final_result = performance_row(final_daily, AUM)
final_result['IC_mean'], final_result['IC_tstat'] = final_ic
final_result['elapsed_seconds'] = round(time.perf_counter() - t0, 2)

pd.Series(final_result)[[
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'gross_sharpe', 'commission_drag', 'slippage_drag', 'borrow_drag',
    'average_gross_exposure_used', 'average_turnover', 'IC_mean', 'IC_tstat',
    'elapsed_seconds',
]]


Loading cached Phase 2 weights from model_cache/phase2_weights_v1_expanding_prior50_learned50_dev20241231.csv...


net_annual_return              0.073099
net_vol                        0.049673
net_sharpe                     1.445318
max_drawdown                  -0.101865
gross_sharpe                   3.111879
commission_drag                0.377699
slippage_drag                  1.136577
borrow_drag                    0.152285
average_gross_exposure_used    0.749713
average_turnover               1.499426
IC_mean                        0.028940
IC_tstat                       9.934090
elapsed_seconds               24.340000
dtype: object

In [10]:
model_comparison = pd.DataFrame([
    {
        'model': 'baseline_rank_target_4y',
        'target': 'cross_sectional_rank(overnight_next)',
        'training_window': f'{config.training_years}y rolling',
        'net_annual_return': baseline_result['net_annual_return'],
        'net_vol': baseline_result['net_vol'],
        'net_sharpe': baseline_result['net_sharpe'],
        'max_drawdown': baseline_result['max_drawdown'],
        'IC_mean': baseline_result['IC_mean'],
        'IC_tstat': baseline_result['IC_tstat'],
    },
    {
        'model': phase2_final_spec.experiment_id,
        'target': 'overnight_next / (vol20 / sqrt(252))',
        'training_window': phase2_final_spec.training_window,
        'net_annual_return': final_result['net_annual_return'],
        'net_vol': final_result['net_vol'],
        'net_sharpe': final_result['net_sharpe'],
        'max_drawdown': final_result['max_drawdown'],
        'IC_mean': final_result['IC_mean'],
        'IC_tstat': final_result['IC_tstat'],
    },
])
model_comparison


,model,target,training_window,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat
0,baseline_rank_target_4y,cross_sectional_rank(overnight_next),4y rolling,0.016869,0.047099,0.378713,-0.245771,0.027312,8.362962
1,phase2_g5_05_expanding,overnight_next / (vol20 / sqrt(252)),expanding,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090


## 6. Reproduce Model Search And Tried Alternatives

The final model was not chosen from a single attempt. This section reruns the archived model-search grids from the current notebook state: Phase 1 target/weighting/borrow/ranking alternatives, then Phase 2 challenger variants including training-window choices, liquidity/volatility weighting, ridge, elastic net, and robust ridge. It writes the two experiment summary CSVs used by the report.

In [11]:
t0 = time.perf_counter()
phase1_rows = []
phase1_scored_by_target = {'rank': baseline_scored}
phase1_ic_by_target = {'rank': baseline_ic}

for spec in experiment_specs():
    print(f"Phase 1 grid: {spec.experiment_group} / {spec.experiment_name}", flush=True)
    if spec.target_transform not in phase1_scored_by_target:
        scored = score_panel_with_target(model_panel, rank_columns, config, spec.target_transform)
        scored = add_walk_forward_alpha_return(scored, config)
        scored = slim_panel(scored.drop(columns=['experiment_target'], errors='ignore'), rank_columns)
        phase1_scored_by_target[spec.target_transform] = scored
        phase1_ic_by_target[spec.target_transform] = ic_summary(scored)
    scored_panel = phase1_scored_by_target[spec.target_transform]
    ic_values = phase1_ic_by_target[spec.target_transform]
    for aum in config.aums:
        daily = run_experiment_backtest(scored_panel, aum, config, spec)
        phase1_rows.append(summarise_experiment(daily, spec, aum, ic_values))

phase1_log = pd.DataFrame(phase1_rows)
phase1_log.to_csv(config.output_dir / 'strategy_experiment_summary.csv', index=False)

phase1_generation_summary = pd.Series({
    'rows_written': len(phase1_log),
    'unique_experiments': phase1_log[['experiment_group', 'experiment_name']].drop_duplicates().shape[0],
    'target_transforms_scored': ', '.join(sorted(phase1_scored_by_target)),
    'output': 'outputs/strategy_experiment_summary.csv',
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
})
phase1_generation_summary

Phase 1 grid: A_basket_size / top_bottom_3pct


Phase 1 grid: A_basket_size / top_bottom_5pct


Phase 1 grid: A_basket_size / top_bottom_7_5pct


Phase 1 grid: A_basket_size / top_bottom_10pct


Phase 1 grid: A_basket_size / long_3pct_short_5pct


...

  scoring vol_scaled target for year 2021...


  scoring vol_scaled target for year 2022...


  scoring vol_scaled target for year 2023...


  scoring vol_scaled target for year 2024...


rows_written                                                         72
unique_experiments                                                   24
target_transforms_scored    demeaned, rank, raw, vol_scaled, winsorised
output                          outputs/strategy_experiment_summary.csv
elapsed_seconds                                              549.200000
dtype: object

In [12]:
def build_notebook_champion_matrix(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for experiment_id, group in summary.groupby('experiment_id', sort=False):
        meta = group.iloc[0][[
            'experiment_id', 'experiment_group', 'experiment_name', 'training_window',
            'alpha_learning_method', 'weighting_scheme', 'ranking_scheme', 'short_treatment', 'notes',
        ]].to_dict()
        row = dict(meta)
        for period in ['validation_2019_2022', 'internal_holdout_2023_2024', 'full_2010_2024']:
            subset = group.loc[group['AUM'].eq(AUM) & group['period'].eq(period)]
            if subset.empty:
                continue
            s = subset.iloc[0]
            prefix = f'{period}_250m'
            row[f'{prefix}_net_sharpe'] = s['net_sharpe']
            row[f'{prefix}_max_drawdown'] = s['max_drawdown']
            row[f'{prefix}_worst_12m_return'] = s['worst_12m_return']
        for aum, label in [(50_000_000.0, '50m'), (250_000_000.0, '250m'), (1_000_000_000.0, '1b')]:
            subset = group.loc[group['AUM'].eq(aum) & group['period'].eq('full_2010_2024')]
            if subset.empty:
                continue
            s = subset.iloc[0]
            row[f'full_{label}_net_sharpe'] = s['net_sharpe']
            if aum == AUM:
                for column in [
                    'average_gross_exposure_used', 'average_turnover', 'commission_drag',
                    'slippage_drag', 'borrow_drag', 'IC_mean', 'IC_tstat',
                ]:
                    row[f'full_250m_{column}'] = s[column]
        rows.append(row)
    matrix = pd.DataFrame(rows)
    matrix['passes_phase2_replacement_rule'] = (
        matrix['validation_2019_2022_250m_net_sharpe'].gt(1.107)
        & matrix['internal_holdout_2023_2024_250m_net_sharpe'].ge(0.0)
        & matrix['full_2010_2024_250m_net_sharpe'].gt(0.811)
    )
    return matrix.sort_values(
        ['passes_phase2_replacement_rule', 'validation_2019_2022_250m_net_sharpe', 'full_2010_2024_250m_net_sharpe'],
        ascending=[False, False, False],
    ).reset_index(drop=True)


t0 = time.perf_counter()
phase2_rows = []
phase2_scored_cache = {phase2_final_spec.scored_key: final_scored}
phase2_ic_cache = {}
phase2_daily_cache = {}

for spec in phase2_specs():
    print(f"Phase 2 grid: {spec.experiment_id} / {spec.experiment_name}", flush=True)
    if spec.scored_key not in phase2_scored_cache:
        phase2_scored_cache[spec.scored_key] = score_panel_phase2(
            model_panel,
            rank_columns,
            config,
            spec.training_window,
            spec.alpha_learning_method,
            use_model_cache=USE_MODEL_CACHE,
        )
    scored = phase2_scored_cache[spec.scored_key]

    for period_name, start, end in PERIODS:
        key = (spec.scored_key, period_name)
        if key not in phase2_ic_cache:
            phase2_ic_cache[key] = ic_for_period(scored, start, end)

    for aum in config.aums:
        daily_key = (spec.selection_key, aum)
        if daily_key not in phase2_daily_cache:
            if spec.experiment_id == phase2_final_spec.experiment_id and aum == AUM:
                phase2_daily_cache[daily_key] = final_daily
            else:
                phase2_daily_cache[daily_key] = run_phase2_backtest(scored, aum, config, spec)
        daily = phase2_daily_cache[daily_key]
        for period_name, start, end in PERIODS:
            phase2_rows.append(
                summarise_period(
                    daily,
                    spec,
                    aum,
                    period_name,
                    start,
                    end,
                    phase2_ic_cache[(spec.scored_key, period_name)],
                )
            )

phase2_log = pd.DataFrame(phase2_rows)
phase2_log.to_csv(config.output_dir / 'phase2_experiment_summary.csv', index=False)
champion_challenger_matrix = build_notebook_champion_matrix(phase2_log)
champion_challenger_matrix.to_csv(report_ready_dir(config) / 'champion_challenger_matrix.csv', index=False)

phase2_generation_summary = pd.Series({
    'rows_written': len(phase2_log),
    'unique_experiments': phase2_log['experiment_id'].nunique(),
    'matrix_rows_written': len(champion_challenger_matrix),
    'scored_model_variants': len(phase2_scored_cache),
    'portfolio_selection_variants': len({spec.selection_key for spec in phase2_specs()}),
    'output': 'outputs/phase2_experiment_summary.csv',
    'matrix_output': 'outputs/report_ready/champion_challenger_matrix.csv',
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
})
phase2_generation_summary

Phase 2 grid: phase2_g1_01_top_bottom_3pct_equal / top/bottom 3%, equal weight


Loading cached Phase 2 weights from model_cache/phase2_weights_v1_4y_rolling_prior50_learned50_dev20241231.csv...


Phase 2 grid: phase2_g1_02_top_bottom_5pct_equal / top/bottom 5%, equal weight


Phase 2 grid: phase2_g1_03_top_bottom_7_5pct_equal / top/bottom 7.5%, equal weight


Phase 2 grid: phase2_g1_04_top_bottom_10pct_equal / top/bottom 10%, equal weight


...

Phase 2 grid: phase2_g6_06_elastic_net_ranked_features / elastic net on ranked features


Loading cached Phase 2 weights from model_cache/phase2_weights_v1_4y_rolling_elastic_net_ranked_features_dev20241231.csv...


Phase 2 grid: phase2_g6_07_robust_ridge_clipped / robust clipped ridge on ranked features


Loading cached Phase 2 weights from model_cache/phase2_weights_v1_4y_rolling_robust_ridge_clipped_dev20241231.csv...


rows_written                                                                  480
unique_experiments                                                             40
matrix_rows_written                                                            40
scored_model_variants                                                          12
portfolio_selection_variants                                                   35
output                                      outputs/phase2_experiment_summary.csv
matrix_output                   outputs/report_ready/champion_challenger_matri...
elapsed_seconds                                                        840.140000
dtype: object

In [13]:
phase1_top_250m = (
    phase1_log.loc[phase1_log['AUM'].eq(AUM)]
    .sort_values('net_sharpe', ascending=False)
    [[
        'experiment_group', 'experiment_name', 'net_annual_return', 'net_vol',
        'net_sharpe', 'max_drawdown', 'average_gross_exposure_used',
        'IC_mean', 'IC_tstat', 'worst_12m_return', 'notes',
    ]]
    .head(10)
    .reset_index(drop=True)
)

phase2_250m = phase2_log.loc[phase2_log['AUM'].eq(AUM)].copy()
meta_cols = [
    'experiment_id', 'experiment_group', 'experiment_name', 'training_window',
    'alpha_learning_method', 'weighting_scheme', 'ranking_scheme', 'short_treatment', 'notes',
]
meta = phase2_250m[meta_cols].drop_duplicates('experiment_id')
metric = phase2_250m.pivot_table(
    index='experiment_id',
    columns='period',
    values='net_sharpe',
    aggfunc='first',
).reset_index()
metric = metric.rename(columns={
    'validation_2019_2022': 'validation_sharpe',
    'internal_holdout_2023_2024': 'holdout_sharpe',
    'full_2010_2024': 'full_sharpe',
})
phase2_selection_table = (
    meta.merge(metric, on='experiment_id', how='left')
    .sort_values(['validation_sharpe', 'holdout_sharpe', 'full_sharpe'], ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print('Top Phase 1 alternatives at 250M, reproduced in this notebook run:')
display(phase1_top_250m)
print('Top Phase 2 challengers at 250M, reproduced in this notebook run:')
display(phase2_selection_table)
print('Final promoted model rows:')
display(
    phase2_log.loc[
        phase2_log['AUM'].eq(AUM)
        & phase2_log['experiment_id'].eq(phase2_final_spec.experiment_id)
        & phase2_log['period'].isin(['validation_2019_2022', 'internal_holdout_2023_2024', 'full_2010_2024'])
    ][['experiment_id', 'period', 'net_sharpe', 'net_annual_return', 'max_drawdown', 'IC_mean', 'IC_tstat']]
)

Top Phase 1 alternatives at 250M, reproduced in this notebook run:


,experiment_group,experiment_name,net_annual_return,net_vol,net_sharpe,max_drawdown,average_gross_exposure_used,IC_mean,IC_tstat,worst_12m_return,notes
0,E_target_transform,volatility_scaled_overnight_return,0.039314,0.049002,0.811479,-0.101865,0.704268,0.028168,9.464596,-0.100903,
1,E_target_transform,demeaned_overnight_return,0.041952,0.055815,0.764228,-0.101865,0.662474,0.026895,7.811144,-0.100903,
2,E_target_transform,winsorised_overnight_return,0.043284,0.057676,0.763528,-0.113368,0.714413,0.028046,8.293189,-0.100903,
3,E_target_transform,raw_overnight_return,0.039073,0.057765,0.692414,-0.118771,0.701422,0.028081,8.381988,-0.110513,
4,B_weighting,score_divided_by_volatility,0.024045,0.050217,0.498271,-0.220742,0.578925,0.027312,8.362962,-0.114101,
5,B_weighting,score_weighted,0.021922,0.051870,0.444001,-0.250220,0.578925,0.027312,8.362962,-0.136480,
6,B_weighting,volatility_weighted,0.018908,0.046111,0.429280,-0.220817,0.578925,0.027312,8.362962,-0.121755,
7,C_cost_aware_ranking,short_alpha_minus_borrow_and_liquidity_penalty,0.019296,0.048464,0.418567,-0.240452,0.583384,0.027312,8.362962,-0.131526,Short ranking penalises borrow tier and liquid...
8,D_short_leg,exclude_tier_c_shorts,0.017349,0.047193,0.388033,-0.242452,0.594226,0.027312,8.362962,-0.128763,
9,E_target_transform,cross_sectional_rank_target,0.016869,0.047099,0.378713,-0.245771,0.578925,0.027312,8.362962,-0.129770,Current baseline target.


Top Phase 2 challengers at 250M, reproduced in this notebook run:


,experiment_id,experiment_group,experiment_name,training_window,alpha_learning_method,weighting_scheme,ranking_scheme,short_treatment,notes,design_2010_2018,full_sharpe,holdout_sharpe,validation_sharpe
0,phase2_g5_05_expanding,G5_training_window,expanding window,expanding,prior50_learned50,equal_weight,raw_alpha_score,tiered_borrow_cost_only,,1.170126,1.445318,0.620163,2.278993
1,phase2_g2_06_score_div_vol_liquidity_floor,G2_weighting_scheme,score divided by volatility with liquidity floor,4y_rolling,prior50_learned50,score_divided_by_volatility_liquidity_floor,raw_alpha_score,tiered_borrow_cost_only,Low-ADV selected names receive lower sizing pr...,1.248001,1.275686,0.818036,1.754876
2,phase2_g2_04_score_divided_by_volatility,G2_weighting_scheme,score divided by volatility,4y_rolling,prior50_learned50,score_divided_by_volatility,raw_alpha_score,tiered_borrow_cost_only,,1.220091,1.241436,0.760509,1.710867
3,phase2_g6_06_elastic_net_ranked_features,G6_transparent_alpha_learning,elastic net on ranked features,4y_rolling,elastic_net_ranked_features,equal_weight,raw_alpha_score,tiered_borrow_cost_only,Uses scikit-learn already present in the recor...,1.361187,1.257566,0.900135,1.571677
4,phase2_g2_03_score_weighted,G2_weighting_scheme,score-weighted,4y_rolling,prior50_learned50,score_weighted,raw_alpha_score,tiered_borrow_cost_only,,1.184593,1.088435,0.372750,1.534315
5,phase2_g6_05_ridge_ranked_features,G6_transparent_alpha_learning,ridge regression on ranked features,4y_rolling,ridge_ranked_features,equal_weight,raw_alpha_score,tiered_borrow_cost_only,,1.325692,1.239191,1.026967,1.523141
6,phase2_g2_05_score_times_liquidity,G2_weighting_scheme,score times liquidity,4y_rolling,prior50_learned50,score_times_liquidity,raw_alpha_score,tiered_borrow_cost_only,,1.291517,1.065721,0.194643,1.499142
7,phase2_g2_07_score_weighted_single_name_cap,G2_weighting_scheme,score-weighted with max single-name cap,4y_rolling,prior50_learned50,score_weighted_single_name_cap,raw_alpha_score,tiered_borrow_cost_only,Adds a 10% per-side sizing cap; participation ...,1.193064,1.040548,0.311993,1.495513
8,phase2_g2_02_volatility_weighted,G2_weighting_scheme,volatility-weighted,4y_rolling,prior50_learned50,volatility_weighted,raw_alpha_score,tiered_borrow_cost_only,,1.104665,1.055630,0.472621,1.429059
9,phase2_g6_04_learned100,G6_transparent_alpha_learning,0% prior / 100% learned,4y_rolling,learned100,equal_weight,raw_alpha_score,tiered_borrow_cost_only,,1.122152,0.992155,0.301491,1.299937


Final promoted model rows:


,experiment_id,period,net_sharpe,net_annual_return,max_drawdown,IC_mean,IC_tstat
377,phase2_g5_05_expanding,validation_2019_2022,2.278993,0.174141,-0.054656,0.019862,3.336067
378,phase2_g5_05_expanding,internal_holdout_2023_2024,0.620163,0.033329,-0.076432,0.017290,2.256992
379,phase2_g5_05_expanding,full_2010_2024,1.445318,0.073099,-0.101865,0.028940,9.934090


## 7. Validation, Holdout, And Stress Windows

The final model is split into design, validation, internal holdout, and full development periods. If the data folder contains post-2024 held-out dates, the notebook also reports a post-development period scored with weights frozen at the development cutoff.

In [14]:
periods_to_report = list(PERIODS)
if config.cutoff_ts > SUBMISSION_DEVELOPMENT_CUTOFF:
    periods_to_report.append(('post_development_after_2024', '2025-01-01', config.cutoff))
    periods_to_report.append((f'full_{config.start_ts.year}_{config.cutoff_ts.year}', config.start_date, config.cutoff))

period_rows = []
for period_name, start, end in periods_to_report:
    period_rows.append(
        summarise_period(
            final_daily,
            phase2_final_spec,
            AUM,
            period_name,
            start,
            end,
            ic_for_period(final_scored, start, end),
        )
    )
period_summary = pd.DataFrame(period_rows)
period_summary[[
    'period', 'start_date', 'end_date', 'net_annual_return', 'net_vol',
    'net_sharpe', 'max_drawdown', 'IC_mean', 'IC_tstat', 'worst_12m_return',
    'average_gross_exposure_used', 'fraction_cap_binding_days',
]]

,period,start_date,end_date,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat,worst_12m_return,average_gross_exposure_used,fraction_cap_binding_days
0,design_2010_2018,2010-01-01,2018-12-31,0.039625,0.033697,1.170126,-0.101865,0.035559,9.630745,-0.100903,0.628556,1.000000
1,validation_2019_2022,2019-01-01,2022-12-31,0.174141,0.071589,2.278993,-0.054656,0.019862,3.336067,0.065084,0.911750,1.000000
2,internal_holdout_2023_2024,2023-01-01,2024-12-31,0.033329,0.055327,0.620163,-0.076432,0.017290,2.256992,-0.075479,0.970761,0.998008
3,full_2010_2024,2010-01-01,2024-12-31,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090,-0.100903,0.749713,0.999735


In [15]:
stress_windows = {
    'late_2018': ('2018-10-01', '2018-12-31'),
    'covid_q1_2020': ('2020-01-01', '2020-03-31'),
    'drawdown_2022': ('2022-01-01', '2022-12-31'),
}

stress_rows = []
final_daily_for_stress = final_daily.copy()
final_daily_for_stress['date'] = pd.to_datetime(final_daily_for_stress['date'])
for name, (start, end) in stress_windows.items():
    chunk = final_daily_for_stress.loc[final_daily_for_stress['date'].between(start, end)]
    stress_rows.append({
        'window': name,
        'start': start,
        'end': end,
        'net_return': (1.0 + chunk['net_return']).prod() - 1.0,
        'net_sharpe': chunk['net_return'].mean() / chunk['net_return'].std(ddof=1) * np.sqrt(252.0),
        'max_drawdown': ((1.0 + chunk['net_return']).cumprod() / (1.0 + chunk['net_return']).cumprod().cummax() - 1.0).min(),
        'avg_gross_exposure': chunk['gross_exposure'].mean(),
        'days': len(chunk),
    })
pd.DataFrame(stress_rows)


,window,start,end,net_return,net_sharpe,max_drawdown,avg_gross_exposure,days
0,late_2018,2018-10-01,2018-12-31,-0.005931,-0.482211,-0.019200,0.852784,63
1,covid_q1_2020,2020-01-01,2020-03-31,0.075478,2.798751,-0.041378,0.889505,62
2,drawdown_2022,2022-01-01,2022-12-31,0.320946,3.524723,-0.026347,0.992267,251


## 8. Feature Ablation And Model Risk

This section reruns the full feature-group ablation used in the report. Each variant retrains the transparent expanding-window final model after removing one feature family, then backtests all required AUM levels with the same costs, borrow treatment, and 5% ADV20 cap.

In [16]:
ablation_variants = [('full_model', 'none', rank_columns)]
for group_name, group_columns in FEATURE_GROUPS.items():
    remaining = [column for column in rank_columns if column not in set(group_columns)]
    ablation_variants.append((f'remove_{group_name}', group_name, remaining))

ablation_rows = []
for variant_name, removed_group, columns in ablation_variants:
    t0 = time.perf_counter()
    if variant_name == 'full_model':
        scored_variant = final_scored
    else:
        scored_variant = score_panel_phase2(
            model_panel,
            columns,
            config,
            phase2_final_spec.training_window,
            phase2_final_spec.alpha_learning_method,
        )
    ic_values = ic_summary(scored_variant)
    elapsed = round(time.perf_counter() - t0, 2)
    for aum in config.aums:
        if variant_name == 'full_model' and aum == AUM:
            daily_variant = final_daily
        else:
            daily_variant = run_experiment_backtest(scored_variant, aum, config, final_spec())
        row = summarise_experiment(daily_variant, final_spec(), aum, ic_values)
        ablation_rows.append({
            'model_variant': variant_name,
            'removed_group': removed_group,
            'AUM': aum,
            'feature_count': len(columns),
            'IC_mean': row['IC_mean'],
            'IC_tstat': row['IC_tstat'],
            'net_annual_return': row['net_annual_return'],
            'net_vol': row['net_vol'],
            'net_sharpe': row['net_sharpe'],
            'max_drawdown': row['max_drawdown'],
            'avg_turnover': row['average_daily_turnover'],
            'avg_gross_exposure_used': row['average_gross_exposure_used'],
            'scoring_elapsed_seconds': elapsed,
        })

ablation_summary = pd.DataFrame(ablation_rows)
ablation_summary.loc[ablation_summary['AUM'].eq(AUM), [
    'model_variant', 'removed_group', 'feature_count', 'net_annual_return',
    'net_vol', 'net_sharpe', 'max_drawdown', 'IC_mean', 'IC_tstat',
    'avg_gross_exposure_used', 'scoring_elapsed_seconds',
]]

  Phase 2 scoring expanding / prior50_learned50 for 2010...


  Phase 2 scoring expanding / prior50_learned50 for 2011...


  Phase 2 scoring expanding / prior50_learned50 for 2012...


  Phase 2 scoring expanding / prior50_learned50 for 2013...


  Phase 2 scoring expanding / prior50_learned50 for 2014...


...

  Phase 2 scoring expanding / prior50_learned50 for 2021...


  Phase 2 scoring expanding / prior50_learned50 for 2022...


  Phase 2 scoring expanding / prior50_learned50 for 2023...


  Phase 2 scoring expanding / prior50_learned50 for 2024...


,model_variant,removed_group,feature_count,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat,avg_gross_exposure_used,scoring_elapsed_seconds
1,full_model,none,28,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090,0.749713,1.690000
4,remove_return/reversal,return/reversal,22,-0.036228,0.032647,-1.113876,-0.466622,0.011262,3.332858,0.476798,18.390000
7,remove_risk/liquidity/size,risk/liquidity/size,23,0.058921,0.051223,1.143509,-0.082442,0.027814,10.863295,0.806179,17.710000
10,remove_fundamental/value/quality,fundamental/value/quality,19,0.070140,0.049025,1.407451,-0.097542,0.029606,10.203432,0.745450,14.710000
13,remove_earnings/revision,earnings/revision,24,0.073096,0.048579,1.476684,-0.101865,0.028372,9.753544,0.748835,16.970000
16,remove_short-interest/borrow-stress,short-interest/borrow-stress,25,0.076977,0.050010,1.508078,-0.113176,0.028908,9.846413,0.783239,16.880000
19,remove_industry-return,industry-return,27,0.072742,0.049557,1.441862,-0.101865,0.028955,9.942310,0.749337,22.490000


## 9. Capacity, Cost, And Position Checks

The final 250M portfolio is generated in this notebook. These checks use the fresh positions and daily returns, not the submitted parquet file.


In [17]:
capacity_cost_summary = pd.Series({
    'days': len(final_daily),
    'positions': len(final_positions),
    'average_gross_exposure': final_daily['gross_exposure'].mean(),
    'fraction_cap_binding_days': final_daily['cap_binding'].mean(),
    'average_turnover': final_daily['turnover'].mean(),
    'average_long_names': final_daily['n_long'].mean(),
    'average_short_names': final_daily['n_short'].mean(),
    'avg_commission_bps_per_day': final_daily['commission_cost'].mean() * 10_000,
    'avg_slippage_bps_per_day': final_daily['slippage_cost'].mean() * 10_000,
    'avg_borrow_bps_per_day': final_daily['borrow_cost'].mean() * 10_000,
    'max_participation_rate': final_positions['participation_rate'].max(),
})
capacity_cost_summary


days                           3,774.000000
positions                    188,070.000000
average_gross_exposure             0.749713
fraction_cap_binding_days          0.999735
average_turnover                   1.499426
average_long_names                24.916534
average_short_names               24.916534
avg_commission_bps_per_day         0.749713
avg_slippage_bps_per_day           2.249139
avg_borrow_bps_per_day             0.300775
max_participation_rate             0.050000
dtype: float64

In [18]:
borrow_tier_summary = (
    final_positions.loc[final_positions['side'].eq('SHORT')]
    .groupby('borrow_tier')
    .agg(
        short_position_days=('borrow_tier', 'size'),
        average_short_notional=('abs_dollar_position', 'mean'),
        total_borrow_cost=('borrow_cost', 'sum'),
    )
    .reset_index()
)
borrow_tier_summary


,borrow_tier,short_position_days,average_short_notional,total_borrow_cost
0,A,40629,"4,352,833.192034","2,807,162.853320"
1,B,35972,"3,568,609.022803","10,188,095.537163"
2,C,17434,"2,779,389.714057","15,382,819.134877"


## 10. Regenerate Submitted Outputs From The Notebook

This is the reproduction step the marker needs. The notebook writes the final daily returns, position files, performance summary, ablation table, capacity table, timing/lag audits, stress windows, and QuantStats HTML directly from the freshly computed model objects above.

If `model_cache/` contains compatible trained Phase 2 weight schedules, the notebook loads those model artefacts and still regenerates positions, returns, report tables, figures, and tests from the raw panel. If the cache is missing or incompatible, it trains the weights and writes a fresh cache.


In [19]:
def build_feature_inventory(rank_columns: list[str]) -> pd.DataFrame:
    rows = [
        ('rank_r_on_1d', 'r_overnight = adj_open_t / adj_close_{t-1} - 1', 'prices.parquet: adjusted open, adjusted close', '0 trading days; Open_t is known after the 09:30 auction', 'Yes'),
        ('rank_r_on_5d', 'rolling mean of r_overnight over 5 sessions, min_periods=3', 'prices.parquet: adjusted open, adjusted close', '0 trading days; uses overnight returns available by 15:50 ET', 'Yes'),
        ('rank_r_id_1d', 'r_intraday_{t-1} = adj_close_{t-1} / adj_open_{t-1} - 1', 'prices.parquet: adjusted open, adjusted close', '1 trading day', 'Yes'),
        ('rank_ret_cc_5d', 'adj_close_{t-1} / adj_close_{t-6} - 1', 'prices.parquet: adjusted close', '1 trading day', 'Yes'),
        ('rank_ret_cc_20d', 'adj_close_{t-1} / adj_close_{t-21} - 1', 'prices.parquet: adjusted close', '1 trading day', 'Yes'),
        ('rank_ret_cc_60d', 'adj_close_{t-1} / adj_close_{t-61} - 1', 'prices.parquet: adjusted close', '1 trading day', 'Yes'),
        ('rank_vol20', 'annualised rolling std of close-to-close returns over 20 sessions, shifted 1 day', 'prices.parquet: adjusted close', '1 trading day', 'Yes'),
        ('rank_vol60', 'annualised rolling std of close-to-close returns over 60 sessions, shifted 1 day', 'prices.parquet: adjusted close', '1 trading day', 'Yes'),
        ('rank_amihud20', 'rolling mean over 20 sessions of abs(r_close_close) / dollar_volume, shifted 1 day', 'prices.parquet: adjusted close, volume', '1 trading day', 'Yes'),
        ('rank_adv20_log', 'log(1 + trailing 20-day average dollar volume), shifted 1 day', 'prices.parquet: adjusted close, volume', '1 trading day', 'Yes'),
        ('rank_market_cap_log', 'log(1 + market_cap_{t-1})', 'prices.parquet: market_cap', '1 trading day', 'Yes'),
        ('rank_piot_norm_lag1', 'piot_norm_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_gross_profit_margin_lag1', 'gross_profit_margin_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_asset_turnover_ratio_lag1', 'asset_turnover_ratio_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_net_debt_to_equity_lag1', 'net_debt_to_equity_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_price_to_book_lag1', 'price_to_book_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_ev_to_ebit_lag1', 'ev_to_ebit_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_sue_lag1', 'sue_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_deps_lag1', 'deps_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_reps1_lag1', 'reps1_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_repsf4_lag1', 'repsf4_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
        ('rank_final_score_lag1', 'final_score_{t-1}', 'cheapness_scores.parquet', '1 trading day', 'Yes'),
        ('rank_score_velocity_lag1', 'score_velocity_{t-1}', 'cheapness_scores.parquet', '1 trading day', 'Yes'),
        ('rank_momentum_score_lag1', 'momentum_score_{t-1}', 'cheapness_scores.parquet', '1 trading day', 'Yes'),
        ('rank_dsi_lag1', 'dsi_{t-1}; daily short-interest proxy after publication/vendor lag already in source', 'all_data.parquet', '1 trading day after source availability', 'Yes, if source obeys Section 2.1.3 lag convention'),
        ('rank_dtcn_lag1', 'dtcn_{t-1}; daily days-to-cover proxy after publication/vendor lag already in source', 'all_data.parquet', '1 trading day after source availability', 'Yes, if source obeys Section 2.1.3 lag convention'),
        ('rank_ddtcn_lag1', 'ddtcn_{t-1}; lagged change/stress proxy after publication/vendor lag already in source', 'all_data.parquet', '1 trading day after source availability', 'Yes, if source obeys Section 2.1.3 lag convention'),
        ('rank_industry_return_lag1', 'industry_return_{t-1}', 'all_data.parquet', '1 trading day', 'Yes'),
    ]
    inventory = pd.DataFrame(rows, columns=['feature_name', 'formula', 'data_source', 'required_lag', 'observable_1550_et_day_t'])
    return inventory.loc[inventory['feature_name'].isin(rank_columns)].reset_index(drop=True)


config.output_dir.mkdir(parents=True, exist_ok=True)
rr_dir = report_ready_dir(config)

prepared.universe_summary.to_csv(config.output_dir / 'universe_summary.csv', index=False)
prepared.reconciliation_summary.to_csv(config.output_dir / 'return_reconciliation_summary.csv', index=False)
return_decomposition.to_csv(config.output_dir / 'equal_weight_return_decomposition.csv', index=False)
ablation_output_columns = [
    'model_variant', 'removed_group', 'AUM', 'IC_mean', 'IC_tstat',
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'avg_turnover', 'avg_gross_exposure_used',
]
ablation_summary[ablation_output_columns].to_csv(config.output_dir / 'feature_ablation_summary.csv', index=False)
build_feature_inventory(rank_columns).to_csv(rr_dir / 'feature_inventory.csv', index=False)

write_core_inputs = pd.Series({
    'universe_summary_rows': len(prepared.universe_summary),
    'return_reconciliation_rows': len(prepared.reconciliation_summary),
    'return_decomposition_rows': len(return_decomposition),
    'feature_ablation_rows': len(ablation_summary),
    'feature_inventory_rows': len(rank_columns),
})
write_core_inputs


def write_final_expanding_feature_weights(model_panel: pd.DataFrame, rank_columns: list[str]) -> pd.DataFrame:
    schedule, _ = load_or_train_phase2_weight_schedule(
        model_panel,
        rank_columns,
        config,
        phase2_final_spec.training_window,
        phase2_final_spec.alpha_learning_method,
        use_model_cache=USE_MODEL_CACHE,
    )
    out = schedule.drop(
        columns=['cache_version', 'feature_count', 'feature_set'],
        errors='ignore',
    )
    out.to_csv(report_ready_dir(config) / 'final_expanding_feature_weights.csv', index=False)
    return out

def write_return_decomposition_plot(decomposition: pd.DataFrame) -> Path:
    out = report_ready_dir(config) / 'return_decomposition_q1.png'
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    plot = decomposition.copy()
    plot['date'] = pd.to_datetime(plot['date'])
    for column, label in [
        ('overnight', 'Overnight'),
        ('intraday', 'Intraday'),
        ('close_to_close', 'Close-to-close'),
    ]:
        ax.plot(plot['date'], (1.0 + plot[column].fillna(0.0)).cumprod(), label=label, linewidth=1.6)
    ax.set_title('Equal-weight universe return decomposition')
    ax.set_ylabel('Growth of $1')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=180)
    plt.close(fig)
    return out


def write_lightgbm_diagnostic_from_notebook() -> pd.DataFrame:
    try:
        from lightgbm_diagnostic import run as run_lightgbm_diagnostic
        return run_lightgbm_diagnostic(ROOT)
    except Exception as exc:
        fallback = pd.DataFrame([
            {
                'model': 'LightGBM',
                'lightgbm_version': '',
                'label': 'diagnostic_unavailable',
                'train_start': '',
                'train_end': '',
                'test_start': '',
                'test_end': '',
                'n_train_rows': np.nan,
                'n_test_rows': 0,
                'daily_ic_count': 0,
                'IC_mean': np.nan,
                'IC_tstat': np.nan,
                'notes': f'LightGBM diagnostic could not run in this environment: {exc}',
            },
            {
                'model': 'Final transparent expanding-window linear score',
                'lightgbm_version': '',
                'label': 'diagnostic_unavailable',
                'train_start': '',
                'train_end': '',
                'test_start': '',
                'test_end': '',
                'n_train_rows': np.nan,
                'n_test_rows': 0,
                'daily_ic_count': 0,
                'IC_mean': np.nan,
                'IC_tstat': np.nan,
                'notes': 'Reference row included so report generation remains explicit about the skipped diagnostic.',
            },
        ])
        fallback.to_csv(report_ready_dir(config) / 'lightgbm_diagnostic.csv', index=False)
        (report_ready_dir(config) / 'lightgbm_diagnostic.md').write_text(
            '# LightGBM Diagnostic\n\nDiagnostic unavailable in this environment.\n',
            encoding='utf-8',
        )
        return fallback


In [20]:
daily_by_aum = {}
positions_by_aum = {}
performance_rows = []

for aum in config.aums:
    if aum == AUM:
        daily, positions = final_daily.copy(), final_positions.copy()
    else:
        daily, positions = final_backtest_with_positions(final_scored, aum, config)
    daily_by_aum[aum] = daily
    positions_by_aum[aum] = positions
    label = aum_label(aum)
    daily.to_csv(config.output_dir / f'daily_returns_{label}.csv', index=False)
    positions.to_parquet(config.output_dir / f'positions_{label}.parquet', index=False)
    performance_rows.append(performance_row(daily, aum))

performance = pd.DataFrame(performance_rows)
performance['IC_mean'], performance['IC_tstat'] = final_ic
performance.to_csv(config.output_dir / 'performance_summary.csv', index=False)

# Caches required by diagnostic tools are also generated by the notebook.
model_panel.to_parquet(cache_path(config, 'panel'), index=False)
final_scored.to_parquet(phase2_cache_path(config, phase2_final_spec.scored_key), index=False)

# Core report/test data.
prepared.universe_summary.to_csv(config.output_dir / 'universe_summary.csv', index=False)
prepared.reconciliation_summary.to_csv(config.output_dir / 'return_reconciliation_summary.csv', index=False)
return_decomposition.to_csv(config.output_dir / 'equal_weight_return_decomposition.csv', index=False)
eligibility_summary.to_csv(config.output_dir / 'eligibility_summary.csv', index=False)
ablation_output_columns = [
    'model_variant', 'removed_group', 'AUM', 'IC_mean', 'IC_tstat',
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'avg_turnover', 'avg_gross_exposure_used',
]
ablation_summary[ablation_output_columns].to_csv(config.output_dir / 'feature_ablation_summary.csv', index=False)
build_feature_inventory(rank_columns).to_csv(report_ready_dir(config) / 'feature_inventory.csv', index=False)
stress_window_summary(daily_by_aum[AUM]).to_csv(config.output_dir / 'stress_windows_250m.csv', index=False)

# IC data and figures.
ic_daily = compute_information_coefficients(final_scored)
ic_daily.to_csv(config.output_dir / 'ic_daily.csv', index=False)
ic_yearly = summarise_ic(ic_daily)
ic_yearly.to_csv(config.output_dir / 'ic_yearly.csv', index=False)
plot_ic(ic_daily, config.figures_dir / 'rolling_ic.png')
plot_equal_weight_decomposition(return_decomposition, config.figures_dir / 'equal_weight_return_decomposition.png')
write_return_decomposition_plot(return_decomposition)

# Report-ready audit tables.
write_position_capacity_summary(config)
write_borrow_reports(config)
borrow_external_validation = write_borrow_external_validation(config)
write_earnings_timing_examples(config)
write_short_interest_lag_examples(config)
write_stylised_fact_audit(config)
write_final_strategy_config(config)
write_locked_strategy_robustness(config)
feature_weights = write_final_expanding_feature_weights(model_panel, rank_columns)

# Optional model-family diagnostic used in the research discussion.
lightgbm_diagnostic = write_lightgbm_diagnostic_from_notebook()

if not SKIP_QUANTSTATS:
    benchmark = load_benchmark_returns(config)
    returns_250m = daily_by_aum[AUM].set_index('date')['net_return']
    generate_tearsheet(
        returns_250m,
        benchmark,
        config.output_dir / 'quantstats_250m.html',
        f'C2O final strategy - notebook run - {int(AUM / 1_000_000)}M AUM',
    )

reproduced_outputs = pd.Series({
    'performance_rows': len(performance),
    'phase1_experiment_rows': len(phase1_log),
    'phase2_experiment_rows': len(phase2_log),
    'champion_challenger_rows': len(champion_challenger_matrix),
    'feature_ablation_rows': len(ablation_summary),
    'eligibility_rows': len(eligibility_summary),
    'ic_daily_rows': len(ic_daily),
    'ic_yearly_rows': len(ic_yearly),
    'borrow_external_validation_rows': len(borrow_external_validation),
    'lightgbm_diagnostic_rows': len(lightgbm_diagnostic),
    'quantstats_exists': (config.output_dir / 'quantstats_250m.html').exists(),
})
reproduced_outputs

Loading cached Phase 2 weights from model_cache/phase2_weights_v1_expanding_prior50_learned50_dev20241231.csv...


performance_rows                      3
phase1_experiment_rows               72
phase2_experiment_rows              480
champion_challenger_rows             40
feature_ablation_rows                21
eligibility_rows                     90
ic_daily_rows                      3773
ic_yearly_rows                       15
borrow_external_validation_rows       2
lightgbm_diagnostic_rows              4
quantstats_exists                  True
dtype: object

## 11. Regenerated Artefact Audit

The generated files are read back from `outputs/` to verify that the serialized artefacts match the in-memory notebook run.

In [21]:
submitted_perf = pd.read_csv(config.output_dir / 'performance_summary.csv')
submitted_daily = pd.read_csv(config.output_dir / 'daily_returns_250m.csv')
submitted_daily['date'] = pd.to_datetime(submitted_daily['date'])
submitted_positions = pd.read_parquet(config.output_dir / 'positions_250m.parquet')

fresh_daily = daily_by_aum[AUM].copy()
fresh_daily['date'] = pd.to_datetime(fresh_daily['date'])
joined_daily = fresh_daily[['date', 'net_return']].merge(
    submitted_daily[['date', 'net_return']],
    on='date',
    suffixes=('_fresh', '_submitted'),
    how='outer',
)
joined_daily['abs_diff'] = (joined_daily['net_return_fresh'] - joined_daily['net_return_submitted']).abs()

submitted_row = submitted_perf.loc[submitted_perf['AUM'].eq(AUM)].iloc[0]
audit = pd.Series({
    'submitted_perf_rows': len(submitted_perf),
    'submitted_daily_rows': len(submitted_daily),
    'fresh_daily_rows': len(fresh_daily),
    'max_abs_daily_net_return_diff': joined_daily['abs_diff'].max(),
    'fresh_net_sharpe': performance.loc[performance['AUM'].eq(AUM), 'net_sharpe'].iloc[0],
    'submitted_net_sharpe': submitted_row['net_sharpe'],
    'fresh_positions': len(positions_by_aum[AUM]),
    'submitted_positions': len(submitted_positions),
    'quantstats_exists': (config.output_dir / 'quantstats_250m.html').exists(),
    'max_output_date': str(submitted_daily['date'].max().date()),
})
assert audit['max_abs_daily_net_return_diff'] < 1e-12
assert abs(audit['fresh_net_sharpe'] - audit['submitted_net_sharpe']) < 1e-12
assert audit['fresh_positions'] == audit['submitted_positions']
audit

submitted_perf_rows                       3
submitted_daily_rows                   3774
fresh_daily_rows                       3774
max_abs_daily_net_return_diff      0.000000
fresh_net_sharpe                   1.445318
submitted_net_sharpe               1.445318
fresh_positions                      188070
submitted_positions                  188070
quantstats_exists                      True
max_output_date                  2024-12-31
dtype: object

## 12. Submitted Tests

The notebook finishes by running tests from inside the notebook. In the original 2024 submission mode it runs the full suite. If later held-out data are present, it runs the deterministic portfolio tests and skips the 2024-only submission cutoff assertions.


In [23]:
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC)
if config.cutoff_ts <= SUBMISSION_DEVELOPMENT_CUTOFF:
    test_args = [sys.executable, '-m', 'pytest', '-q']
else:
    test_args = [sys.executable, '-m', 'pytest', '-q', 'tests/test_portfolio.py']
    print('Held-out data mode: skipping tests/test_submission_rules.py because those assertions intentionally require no dates after 2024-12-31.')

result = subprocess.run(
    test_args,
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)
assert result.returncode == 0

.............                                                            [100%]
13 passed in 0.81s



## Research Conclusion

The notebook-run evidence supports the promoted final strategy over the baseline inside the 2010-2024 development window. For later data, the same notebook can score the held-out dates using weights frozen at the 2024 development cutoff. The main material risk remains feature concentration: the return/reversal group is highly important, so the final report treats it as a model-dependence risk rather than a universally robust alpha claim.